In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import os

# ── configurações ──────────────────────────────────────────────────────────
base_path  = r'C:\Users\pedro.negreiro\Documents\stack-ensemble-prediction\results\experimento_03_detalhado'
janela     = 15   # muda para 5 ou 10 nas outras células

modelos = {
    'SVM':                'svm',
    'KNN':                'knn',
    'Extra Trees':        'extra_trees',
    'Regressao Logistica':'regressao_logistica',
    'LightGBM':           'lightgbm',
}

cores = {
    'SVM':                '#1f77b4',
    'KNN':                '#ff7f0e',
    'Extra Trees':        '#2ca02c',
    'Regressao Logistica':'#d62728',
    'LightGBM':           '#9467bd',
}

temporadas = [
    '2008-2009', '2009-2010', '2011-2012',
    '2012-2013', '2013-2014', '2014-2015',
    '2015-2016', '2016-2017', '2018-2019', '2019-2020',
    '2020-2021', '2021-2022', '2022-2023', '2023-2024'
]

# ── carrega todos os CSVs detalhados ───────────────────────────────────────
dfs = []
for nome_modelo, arquivo_modelo in modelos.items():
    path = os.path.join(base_path, f'{arquivo_modelo}_experimento_03_predicoes_detalhadas.csv')
    if not os.path.exists(path):
        print(f'Arquivo não encontrado: {path}')
        continue
    df = pd.read_csv(path)
    df['Modelo'] = nome_modelo
    dfs.append(df)

dados = pd.concat(dfs, ignore_index=True)
dados = dados[dados['Janela Incremental'] == janela]

# ── função de plot — 2 temporadas por célula ──────────────────────────────
def plot_curva_aprendizado(temporadas_par, janela):
    fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharey=True)
    fig.suptitle(f'Curva de aprendizado durante a temporada | Janela = {janela}', fontsize=13)

    for ax, temporada in zip(axes, temporadas_par):
        subset = dados[dados['Temporada'] == temporada]

        for nome_modelo in modelos.keys():
            df_modelo = (
                subset[subset['Modelo'] == nome_modelo]
                .sort_values('Ordem Jogo Temporada')
            )
            if df_modelo.empty:
                continue

            ax.plot(
                df_modelo['Ordem Jogo Temporada'],
                df_modelo['Acuracia Acumulada'],
                color=cores[nome_modelo],
                linewidth=1.5,
                label=nome_modelo,
                alpha=0.85
            )

        # linha de referência — acurácia do vanilla (time da casa ganha ~60%)
        ax.axhline(y=0.60, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Baseline (0.60)')

        total_jogos = subset['Ordem Jogo Temporada'].max()
        ax.set_title(f'{temporada}  ({int(total_jogos)} jogos)', fontsize=11)
        ax.set_xlabel('Número do jogo na temporada', fontsize=10)
        ax.set_ylabel('Acurácia acumulada', fontsize=10)
        ax.set_ylim(0.40, 1.00)
        ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))
        ax.grid(axis='y', linestyle='--', alpha=0.4)
        ax.legend(fontsize=8, loc='upper right')

    plt.tight_layout()
    plt.show()